# Deployment

To use the model on new data, we define a function 'predict_next_day' built on `src/predict.py`:

| Entry point | Use case |
|---|---|
| `predict_next_day()` (Python) | Other code or notebooks |

[`docs/05_deployment.md`](../docs/05_deployment.md)

In [ ]:
""" Notebook setup: make the project's modules (config.py, src/) importable """
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if "TF_CPP_MIN_LOG_LEVEL" not in os.environ:
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow info/warning logs

In [ ]:
import json
import subprocess

import pandas as pd
import matplotlib.pyplot as plt

import config
from src.notebook import show_source
from src.data_loader import load_stock_data
from src.predict import load_artifacts, predict_next_day
from src import plotting

plotting.set_plot_style()

## Choose the model to deploy
We take the neural network with the lowest **validation loss** (model selection must never use the test set) from `artifacts/metrics.csv` (written by notebook 04).

In [ ]:
metrics = pd.read_csv(config.METRICS_PATH, index_col=0)
models = metrics[metrics["kind"] == "model"]
best_run = models["val_loss"].idxmin()
best_dir = config.ARTIFACTS_DIR / best_run

print("Deploying run:", best_run)
print("Artifact files:", sorted(p.name for p in best_dir.iterdir()))

**The model file alone is not enough.** To make a correct prediction we must repeat the training time preprocessing exactly with the same features, the same lookback and the same fitted scalers. `metadata.json` records those settings:

In [ ]:
print(json.dumps(json.loads((best_dir / "metadata.json").read_text()), indent=2))

## Predict the next trading day

In [ ]:
""" Inspect predict_next_day """
show_source(predict_next_day)

In [ ]:
""" Predict the next trading day """
artifacts = load_artifacts(best_dir)                      # model + scalers + metadata
df = load_stock_data(start_date=None, verbose=False)      # the latest available data
prediction = predict_next_day(artifacts, df)
pd.Series(prediction).to_frame("value")

In [ ]:
plotting.plot_recent_with_prediction(df, prediction, days=120)
plt.show()

## Simulating daily use
In production the job runs every day after the close, using only data available up to that day. Here we replay the last 20 trading days: for each day we cut the data at that date, predict, and compare with what really happened next.

In [ ]:
rows = []
for as_of in df.index[-21:-1]:
    history = df.loc[:as_of]                                   # only data known on that day
    p = predict_next_day(artifacts, history)
    actual_next = df["Close"].iloc[df.index.get_loc(as_of) + 1]
    rows.append({"as of": as_of.date(), "last close": p["last_close"], "predicted next": p["predicted_close"],
                 "actual next": actual_next, "error ($)": p["predicted_close"] - actual_next})
simulation = pd.DataFrame(rows).set_index("as of")
print(f"MAE over these {len(simulation)} days: ${simulation['error ($)'].abs().mean():.2f}")
simulation.round(2)


## Operating the model (after deployment)
A deployed model needs care:
- **Data quality checks** on every new file. The loader already validates it and catches swapped columns.
- **Performance monitoring:** log each prediction, then compare it with the actual close the next day. Track a rolling MAE against the naive baseline.
- **Drift monitoring:** watch whether feature distributions (volatility, volume) move far from the training ranges.
- **Retraining:** on a schedule (for example monthly) or when monitored error degrades, retrain with the newest data and redo the evaluation before replacing the model.
- **Versioning:** keep old artifact folders so is possible to roll back.

That closes the CRISP-DM loop: monitoring results feed back into **Business Understanding** for the next iteration.